# Olefin Hydroformylation Agent — Results Analysis

This notebook visualises the performance of the agentic LLM optimizer across experimental iterations.  
Run all cells top-to-bottom after at least one complete agent run:
```bash
python src/agent_controller.py
```

**Final Project for CSC 7644: Applied LLM Development**  
Author: Chidera C. Nnadiekwe

In [ ]:
# Import necessary libraries
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import numpy as np

matplotlib.rcParams['figure.dpi'] = 120

# Make src/ and scripts/ importable
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

# load the flat list structure directly
LOG_PATH = PROJECT_ROOT / 'data' / 'experiment_log.json'

with open(LOG_PATH) as f:
    log = json.load(f)  # list of run records

print(f'Loaded {len(log)} experimental run(s).')
print('Keys in first record:', list(log[0].keys()) if log else 'N/A')

## 1. Tabular Summary of All Runs

In [ ]:
# Convert the list of run records into a DataFrame
rows = []
for run in log:
    row = {'iteration': run['iteration']}
    row.update(run['conditions'])
    row.update(run['outcomes'])
    rows.append(row)

# Create DataFrame and set iteration as index
df = pd.DataFrame(rows)
df = df.set_index('iteration')

# l_b_ratio and conversion_pct (not lb_ratio / aldehyde_yield_pct)
highlight_cols = [c for c in ['l_b_ratio', 'conversion_pct', 'ton'] if c in df.columns]
df.style.highlight_max(subset=highlight_cols, color='#d4edda')

## 2. L:B Selectivity Over Iterations

In [ ]:
# Plot L:B ratio convergence
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df.index, df['l_b_ratio'], 'o-', color='#2563eb', linewidth=2,
        markersize=7, label='L:B ratio')
ax.axhline(y=df['l_b_ratio'].max(), color='#2563eb', linestyle='--', alpha=0.4,
           label=f'Best: {df["l_b_ratio"].max():.2f}')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('L:B Selectivity Ratio', fontsize=12)
ax.set_title('Linear-to-Branch Selectivity vs Iteration', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/lb_convergence.png', dpi=150)
plt.show()
print('Saved to results/figures/lb_convergence.png')

## 3. % Aldehyde Yield Over Iterations

In [ ]:
# Plot conversion convergence
fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(df.index, df['conversion_pct'], alpha=0.15, color='#16a34a')
ax.plot(df.index, df['conversion_pct'], 'o-', color='#16a34a', linewidth=2,
        markersize=7, label='Conversion (%)')
ax.set_ylim(0, 100)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Conversion (%)', fontsize=12)
ax.set_title('Substrate Conversion vs Iteration', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/yield_convergence.png', dpi=150)
plt.show()
print('Saved to results/figures/yield_convergence.png')

## 4. Parameter Exploration scatter Plots

In [ ]:
# Scatter plots of L:B ratio vs key parameters
params   = ['temperature_C', 'pressure_bar', 'ligand_loading_eq']
xlabels  = ['Temperature (°C)', 'Pressure (bar)', 'Ligand Equiv']
colors   = ['#7c3aed', '#0891b2', '#ea580c']

valid_params = [(p, xl, c) for p, xl, c in zip(params, xlabels, colors) if p in df.columns]

fig, axes = plt.subplots(1, len(valid_params), figsize=(5 * len(valid_params), 4))
if len(valid_params) == 1:
    axes = [axes]

# Create scatter plots for each valid parameter
for ax, (param, xlabel, color) in zip(axes, valid_params):
    sc = ax.scatter(df[param], df['l_b_ratio'], c=df.index,
                    cmap='viridis', s=80, alpha=0.85, edgecolors='k', linewidths=0.5)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('L:B Ratio', fontsize=11)
    ax.set_title(f'{xlabel} vs L:B', fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.colorbar(sc, ax=ax, label='Iteration')

plt.suptitle('Parameter-Selectivity Relationships', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/parameter_lb_scatter.png', dpi=150)
plt.show()
print('Saved to results/figures/parameter_lb_scatter.png')

## 5. Agent Reasoning Traces

In [ ]:
# Print reasoning snippets for each iteration
for run in log:
    outcomes = run.get('outcomes', {})
    print(f"=== Iteration {run['iteration']} ===")
    print(f"  L:B: {outcomes.get('l_b_ratio', 'N/A')}  "
          f"Conversion: {outcomes.get('conversion_pct', 'N/A')}%  "
          f"TON: {outcomes.get('ton', 'N/A')}")
    reasoning = run.get('reasoning', 'No reasoning available.')
    print(f"  Reasoning (first 400 chars):")
    print(f"    {reasoning[:400]}")
    print()

## 6. Quantitative Summary Metrics

In [ ]:
# Import the evaluation module for consistent metric computation
from evaluation import compute_metrics, generate_random_baseline, compare_agent_vs_baseline

agent_metrics    = compute_metrics(log)
baseline_history = generate_random_baseline(len(log))
baseline_metrics = compute_metrics(baseline_history)

print('\n=== AGENT SUMMARY ===')
print(f"Total iterations: {agent_metrics['total_runs']}")
print(f"Best L:B ratio  : {agent_metrics['max_l_b_ratio']:.3f} "
      f"(iteration {agent_metrics['max_l_b_iteration']})")
print(f"Best conversion : {agent_metrics['max_conversion_pct']:.1f}%")
print(f"Average L:B     : {agent_metrics['avg_l_b_ratio']:.3f} "
      f"± {pd.Series(agent_metrics['l_b_progression']).std():.3f}")
print(f"Average TON     : {agent_metrics['avg_ton']:.1f}")
print(f"Convergence iter: {agent_metrics['convergence_iteration']}")

for threshold in [3.0, 4.0, 5.0]:
    prog = agent_metrics['l_b_progression']
    hits = [i+1 for i, lb in enumerate(prog) if lb >= threshold]
    if hits:
        print(f"First iteration with L:B >= {threshold}: {hits[0]}")
    else:
        print(f"L:B >= {threshold} not achieved in {len(prog)} iterations")

## 7. Agent vs Random Baseline Comparison Plot

In [ ]:
# Compare agent vs baseline
agent_lb    = agent_metrics['l_b_progression']
baseline_lb = baseline_metrics['l_b_progression']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# L:B comparison
axes[0].plot(range(1, len(agent_lb)+1), agent_lb,
             'o-', color='#2563eb', lw=2, ms=7, label='Agent')
axes[0].plot(range(1, len(baseline_lb)+1), baseline_lb,
             's--', color='#dc2626', lw=1.5, ms=5, alpha=0.7, label='Random Baseline')
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('L:B Ratio', fontsize=12)
axes[0].set_title('L:B Ratio: Agent vs Baseline', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Conversion comparison
agent_conv    = agent_metrics['conversion_progression']
baseline_conv = baseline_metrics['conversion_progression']
axes[1].plot(range(1, len(agent_conv)+1), agent_conv,
             'o-', color='#16a34a', lw=2, ms=7, label='Agent')
axes[1].plot(range(1, len(baseline_conv)+1), baseline_conv,
             's--', color='#dc2626', lw=1.5, ms=5, alpha=0.7, label='Random Baseline')
axes[1].set_ylim(0, 100)
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('Conversion (%)', fontsize=12)
axes[1].set_title('Conversion: Agent vs Baseline', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Agent vs Random Baseline Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/agent_vs_baseline.png', dpi=150)
plt.show()
print('Saved to results/figures/agent_vs_baseline.png')